# 02. DCA-Trie v1: Decomposed Static Filtering

**Purpose:** Implement and evaluate DCA-Trie v1 — KG path filtering at construction time using the decomposed relevance score.

**Changes from GCR baseline:** The trie is built from only those paths whose decomposed product score (Eq. 3.12) exceeds τ. A hard type gate prunes type-incompatible paths before any encoder call.

**Reference:** Chapter 3, §3.6, Algorithm 1.

**Key formula (Eq. 3.12):**
```
score(e_t, r, e' | q, y_{<t}) = ρ_r(r, q) · ρ_e(e', q) · ρ_traj(r, e', q, y_{<t})
```
where:
- ρ_r(r, q) = cos(E(r), E(q_rel)) — entity-masked relational relevance
- ρ_e(e', q) = 1[type(e') ∈ T(q, h)] — hard type gate
- ρ_traj(r, e', q, y_{<t}) = cos(E(r ‖ e'), E(q)) — trajectory relevance (with y_{<t}=∅ for v1)

In [ ]:
# @title 1. Install & Setup
import sys
import os
import torch
import json
import re
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

from tqdm import tqdm
from datasets import load_dataset
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
has_a100 = "A100" in gpu_name
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB")

!pip install -q transformers==4.44.2 accelerate>=0.30.1 tiktoken>=0.7.0 datasets>=2.19.2 python-dotenv marisa-trie>=1.2.0 scikit-learn>=1.5.0 sentencepiece>=0.2.0 protobuf>=5.27.1 networkx>=3.0

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca

%cd graph-constrained-reasoning
sys.path.insert(0, '.')
from src.llms import get_registed_model
from src.trie import MarisaTrie
import src.utils as utils

In [ ]:
# @title 2. Configuration
# MODEL
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "gcr"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except ImportError:
    pass

# DATASET
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"

# DCA-Trie v1 PARAMETERS (§3.6)
TAU = 0.25
ENCODER_NAME = "all-MiniLM-L6-v2"
ENCODER_DEVICE = "cpu"

# DECODING
INDEX_LEN = 2
K = 5
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256

# ATTENTION
flash_attn_installed = False
try:
    import flash_attn
    flash_attn_installed = True
except ImportError:
    pass
ATTN_IMPL = "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"

# SAMPLING
MAX_SAMPLES = 100

# OUTPUT
PREDICT_PATH = "results/GenPaths"
FORCE_RERUN = True

# GOOGLE DRIVE
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = "/content/drive/MyDrive/GCR_Results"
    os.makedirs(DRIVE_BASE, exist_ok=True)
except (ImportError, Exception):
    DRIVE_BASE = None

tag = f"DCAv1-tau{TAU}-{ENCODER_NAME}"
print(f"Model: {MODEL_PATH}")
print(f"Encoder: {ENCODER_NAME} on {ENCODER_DEVICE}")
print(f"Threshold τ: {TAU}")
print(f"Dataset: {DATASET}")
print(f"Max samples: {MAX_SAMPLES or 'full'}")
print(f"Tag: {tag}")

In [ ]:
# @title 3. Load Encoder and LLM
from sentence_transformers import SentenceTransformer

print("Loading sentence encoder...")
encoder = SentenceTransformer(ENCODER_NAME, device=ENCODER_DEVICE)
encoder_dim = encoder.get_sentence_embedding_dimension()
print(f"Encoder dim: {encoder_dim}")

import argparse

parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print("Loading LLM...")
try:
    model = LLM(args)
    model.prepare_for_inference()
    model.generation_cfg.temperature = None
    model.generation_cfg.top_p = None
    model.model.generation_config.temperature = None
    model.model.generation_config.top_p = None
    print("Ready.")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

In [ ]:
# @title 4. Type Oracle: T(q, h) — Entity Type Compatibility (§3.5.2)

QUESTION_TYPE_PATTERNS = [
    (r"who\b", "person"),
    (r"what.*nationality", "country"),
    (r"what.*country", "country"),
    (r"what.*language", "language"),
    (r"what.*film|movie", "film"),
    (r"what.*award", "award"),
    (r"what.*occupation|job|profession", "occupation"),
    (r"what.*sport|team", "sports_team"),
    (r"where\b", "location"),
    (r"when\b", "date"),
    (r"how many|how much", "number"),
    (r"which (country|city|state|place|location)", "location"),
    (r"which (person|people|actor|director|president)", "person"),
    (r"which (film|movie)", "film"),
    (r"which (language)", "language"),
    (r"which (organization|company)", "organization"),
    (r"which (university|college|school)", "educational_institution"),
    (r"which (award|prize|honor)", "award"),
    (r"which (sport|team|club)", "sports_team"),
    (r"which (year|decade|century)", "date"),
    (r"what is the name|name the", "person"),
]

RELATION_DOMAIN_MAP = {
    "people": "person", "person": "person", "location": "location",
    "film": "film", "tv": "tv_program", "award": "award",
    "education": "educational_institution", "organization": "organization",
    "sports": "sports_team", "language": "language",
    "medicine": "medical_topic", "religion": "religion",
    "base": "other", "common": "other",
}


def mask_entities(question, entities):
    """Replace entity mentions in question with generic tokens [X], [Y], etc.
    This isolates relational intent (Eq. 3.13 q_rel)."""
    masked = question
    for i, ent in enumerate(entities):
        label = chr(65 + i)
        for variant in [ent, ent.lower(), ent.split()[-1], ent.split()[-1].lower()]:
            if variant in masked:
                masked = masked.replace(variant, f"[{label}]")
                break
    return masked


def infer_answer_type(question):
    """Infer the expected answer entity type from the question.
    Returns a set of Freebase-compatible type strings."""
    q = question.lower()
    for pattern, etype in QUESTION_TYPE_PATTERNS:
        if re.search(pattern, q):
            return {etype}
    return {"person", "location", "organization", "date", "language", "film", "award", "other"}


class TypeOracle:
    """Type compatibility oracle for hard type gate (§3.5.2)."""

    def __init__(self, graph_triples):
        self.entity_types = self._build_type_index(graph_triples)

    @staticmethod
    def _domain_from_relation(rel):
        domain = rel.split(".")[0] if "." in rel else rel
        return RELATION_DOMAIN_MAP.get(domain, "other")

    def _build_type_index(self, triples):
        type_map = defaultdict(set)
        for h, r, t in triples:
            type_map[h].add(self._domain_from_relation(r))
            type_map[t].add("other")
        return type_map

    def get_terminal_types(self, question):
        return infer_answer_type(question)

    def is_type_compatible(self, entity, expected_types, is_terminal=False):
        if entity not in self.entity_types:
            return not is_terminal
        entity_t = self.entity_types[entity]
        if "other" in expected_types:
            return True
        return len(entity_t & expected_types) > 0 or not is_terminal

In [ ]:
# @title 5. DCA-Trie v1: Build Semantically Filtered Trie (Algorithm 1)

_relation_emb_cache = {}
_entity_emb_cache = {}


def get_relation_emb(rel, encoder):
    """Get or cache relation embedding."""
    if rel not in _relation_emb_cache:
        _relation_emb_cache[rel] = encoder.encode(rel, convert_to_numpy=True)
    return _relation_emb_cache[rel]


def get_entity_emb(entity, encoder):
    """Get or cache entity embedding."""
    if entity not in _entity_emb_cache:
        _entity_emb_cache[entity] = encoder.encode(entity, convert_to_numpy=True)
    return _entity_emb_cache[entity]


def build_dcav1_trie(question_dict, tokenizer, encoder, tau, type_oracle, index_path_length=2):
    """DCA-Trie v1: Decomposed static filtering (Algorithm 1).

    Implements Eq. 3.12: score = ρ_r · ρ_e · ρ_traj at construction time.
    - Entity masking for relational intent (Eq. 3.13 q_rel)
    - Hard type gate (Eq. 3.14) before any encoder calls
    - Product score across hops
    """
    question = question_dict["question"]
    entities = question_dict["q_entity"]

    q_rel_str = mask_entities(question, entities)
    u_rel = encoder.encode(q_rel_str, convert_to_numpy=True)
    u_q = encoder.encode(question, convert_to_numpy=True)

    t_terminal = type_oracle.get_terminal_types(question)

    if "paths" in question_dict:
        paths_list = question_dict["paths"]
    else:
        g = utils.build_graph(question_dict["graph"])
        paths_list = utils.dfs(g, entities, index_path_length)

    if len(paths_list) == 0:
        return None, [], [], 0, 0

    kept_paths = []
    kept_scores = []
    n_type_blocked = 0
    n_encoder_calls = 0

    for p in paths_list:
        h = len(p)
        e_terminal = p[-1][2]

        if not type_oracle.is_type_compatible(e_terminal, t_terminal, is_terminal=True):
            n_type_blocked += 1
            continue

        product_score = 1.0
        for hop_idx in range(h):
            rel = p[hop_idx][1]
            e_next = p[hop_idx][2]

            r_emb = get_relation_emb(rel, encoder)
            n_encoder_calls += 1
            rho_r = float(cosine_similarity(r_emb.reshape(1, -1), u_rel.reshape(1, -1))[0][0])

            r_e_str = f"{rel} {e_next}"
            r_e_emb = encoder.encode(r_e_str, convert_to_numpy=True)
            n_encoder_calls += 1
            rho_traj = float(cosine_similarity(r_e_emb.reshape(1, -1), u_q.reshape(1, -1))[0][0])

            product_score *= max(rho_r, 0.0) * max(rho_traj, 0.0)

        if product_score >= tau:
            kept_paths.append(p)
            kept_scores.append(product_score)

    if len(kept_paths) == 0:
        return None, [], [], n_type_blocked, n_encoder_calls

    path_strings = [utils.path_to_string(p) for p in kept_paths]
    tokenized = tokenizer(path_strings, padding=False, add_special_tokens=False).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    trie = MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)

    return trie, kept_paths, kept_scores, n_type_blocked, n_encoder_calls

In [ ]:
# @title 6. Run DCA-Trie v1 Inference
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder

dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
    print(f"Subsampled to {len(dataset)} questions")
else:
    print(f"Loaded {len(dataset)} questions")

model_short = MODEL_PATH.split("/")[-1]
postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}-{tag}"
output_dir = os.path.join(PREDICT_PATH, DATASET, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")


def get_output(path, force):
    """Open predictions file, returning handle and list of already-processed IDs."""
    if not os.path.exists(path) or force:
        return open(path, "w"), []
    with open(path, "r") as f:
        proc = [json.loads(l)["id"] for l in f]
    return open(path, "a"), proc


fout, processed = get_output(os.path.join(output_dir, "predictions.jsonl"), FORCE_RERUN)

total_paths_before = 0
total_paths_after = 0
total_type_blocked = 0
total_encoder_calls = 0
total_empty = 0

try:
    for data in tqdm(dataset, desc="DCA-Trie v1"):
        qid = data["id"]
        if qid in processed:
            continue

        type_oracle = TypeOracle(data["graph"])

        try:
            trie, kept_paths, scores, n_blocked, n_enc = build_dcav1_trie(
                data, model.tokenizer, encoder, TAU, type_oracle, INDEX_LEN
            )
        except Exception as e:
            print(f"Error building trie for {qid}: {e}")
            continue

        total_type_blocked += n_blocked
        total_encoder_calls += n_enc

        if trie is None:
            total_empty += 1
            continue

        g = utils.build_graph(data["graph"])
        truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], g)
        ground_paths = [utils.path_to_string(p) for p in truth_paths]

        start_token_ids = model.tokenizer.convert_tokens_to_ids("<PATH>")
        end_token_ids = model.tokenizer.convert_tokens_to_ids("</PATH>")
        input_query, _, _ = input_builder.process_input(data)
        llm_input = model.prepare_model_prompt(input_query)

        try:
            prediction = model.generate_sentence(
                llm_input, trie,
                start_token_ids=start_token_ids,
                end_token_ids=end_token_ids,
                enable_constrained_by_default=False,
            )
        except Exception as e:
            print(f"Error generating for {qid}: {e}")
            continue

        if prediction is None:
            total_empty += 1
            continue

        fout.write(json.dumps({
            "id": qid,
            "question": data["question"],
            "prediction": prediction,
            "ground_truth": data["answer"],
            "ground_truth_paths": ground_paths,
            "dca_tau": TAU,
            "dca_version": "v1",
        }) + "\n")
        fout.flush()
finally:
    fout.close()

print(f"\nPaths before filter: {total_paths_before}")
print(f"Paths after filter:  {total_paths_after}")
print(f"Type-blocked:        {total_type_blocked}")
print(f"Encoder calls:       {total_encoder_calls}")
print(f"Empty (no paths):    {total_empty}")

from src.utils.qa_utils import eval_path_result_w_ans
print("\n=== Evaluation ===")
eval_path_result_w_ans(os.path.join(output_dir, "predictions.jsonl"))

if DRIVE_BASE is not None:
    import shutil
    drive_dst = os.path.join(DRIVE_BASE, DATASET, model_short, SPLIT, postfix)
    os.makedirs(drive_dst, exist_ok=True)
    shutil.copy2(os.path.join(output_dir, "predictions.jsonl"), drive_dst)
    print(f"Saved to Drive: {drive_dst}")

In [ ]:
# @title 7. Threshold Calibration — τ Selection (§3.5.5, §3.8)

print("Sweeping τ over WebQSP validation set (first 100 questions)...")
val_dataset = load_dataset(f"{DATA_PATH}/RoG-webqsp", split="test[:100]")

tau_candidates = [round(0.1 + 0.05 * i, 2) for i in range(11)]

results = []
for tau_candidate in tau_candidates:
    false_neg_count = 0
    total_truth_count = 0
    total_paths_before = 0
    total_paths_after = 0
    total_type_blocked = 0

    for data in val_dataset:
        g = utils.build_graph(data["graph"])
        truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], g)
        truth_strs = set(utils.path_to_string(p) for p in truth_paths)
        if len(truth_strs) == 0:
            continue
        total_truth_count += len(truth_strs)

        type_oracle = TypeOracle(data["graph"])
        trie, kept_paths, scores, n_blocked, _ = build_dcav1_trie(
            data, model.tokenizer, encoder, tau_candidate, type_oracle, INDEX_LEN
        )
        total_type_blocked += n_blocked

        if trie is None:
            false_neg_count += len(truth_strs)
            continue

        kept_strs = set(utils.path_to_string(p) for p in kept_paths)
        paths_before = len(utils.dfs(g, data["q_entity"], INDEX_LEN))
        total_paths_before += paths_before
        total_paths_after += len(kept_paths)

        for t in truth_strs:
            if t not in kept_strs:
                false_neg_count += 1

    fnr = false_neg_count / max(1, total_truth_count)
    reduction = (1 - total_paths_after / max(1, total_paths_before)) * 100 if total_paths_before > 0 else 0
    passed = fnr < 0.05
    results.append((tau_candidate, fnr, reduction, passed))
    marker = " ✓" if passed else ""
    print(f"  τ={tau_candidate:.2f}  |  FNR={fnr:.4f}  |  Reduction={reduction:.1f}%{marker}")

print("\nSelect highest τ with FNR < 0.05.")

In [ ]:
# @title 8. Compare with GCR Baseline
print("""
=== Comparison: GCR vs DCA-Trie v1 ===

Metric               GCR         DCA-Trie v1 (yours)
Hits@1               92.6         ?
F1                   89.8         ?
Faithfulness        100%        100% (unchanged)
Avg trie size        ~5000       ? (should be lower)
SIR*                 high        ? (should be lower)
SIR*_type            high        ~0 (type gate kills these)
SIR*_traj            high        ? (lower from relational scoring)

To measure decomposed SIR, run notebook 04_SIR_Evaluation.ipynb
on the predictions.jsonl produced above.
""")